# Semi-automated scan reorientation workflow

Author: Zongyu Li

## Workflow overview

1. **Batch QC visualization**  
   Generate a quick-look figure for every scan in a folder. Each figure shows the **center slice in axial, coronal, and sagittal views in one row**, together with the scan name. Both **PNG** and **PDF** files are saved so colleagues can review them easily.

2. **Identify problematic scans**  
   Review the saved figures and identify scans whose orientation does not match the template or the majority of scans.

3. **Interactive manual reorientation**  
   Copy only the problematic scans into a dedicated folder, then use the interactive viewer below to flip them while comparing against a template. The corrected scans are saved using the **template header/affine**.



## Step 1. Batch visualization for orientation QC

This step creates **paged contact-sheet figures** for scan subjects in the path:

- **20 subjects per figure** by default
- **1 subject per row**
- **3 columns per row**
  - axial center slice
  - coronal center slice
  - sagittal center slice

So each saved figure is a **long stacked image** containing up to 20 subjects.
If there are more than 20 subjects, the code automatically continues to the **next figure/page**.

For every page, the code will:

- **save one PNG** file
- **save one PDF** file
- **directly display the figure** in the notebook



In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Step 1: Export paged center-slice QC figures for all scans in a folder
=====================================================================

Purpose
-------
Create paged QC figures in which:
1) each subject occupies one row
2) each row contains three center slices
   - axial
   - coronal
   - sagittal
3) each figure contains up to 20 subjects by default

Each figure is saved as both PNG and PDF, and is also displayed directly
in the notebook. This makes it convenient to review a folder in batches.

This step is intentionally simple and shareable:
- paths are defined at the top of the cell
- output figures are easy to review with colleagues
- it is suitable for identifying scans that should be moved into a
  separate "problematic scans" folder for manual reorientation later
"""

from pathlib import Path
import math
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

# =========================
# Paths to edit
# =========================
SCAN_DIR = Path("/path/to/files/to/inspect")
OUTPUT_FIG_DIR = Path("/path/to/output/qc_figures")
FILE_PATTERN = "*.nii.gz*"

# =========================
# Display and export settings
# =========================
SUBJECTS_PER_FIGURE = 20      # number of subject rows per saved/displayed figure
DOWNSAMPLE = 1                # increase to 2 for faster plotting if images are large
ROW_HEIGHT = 2.2              # inches per subject row
FIGURE_WIDTH = 12             # total width of each saved/displayed figure
TITLE_FONTSIZE = 13
COLUMN_TITLE_FONTSIZE = 11
ROW_LABEL_FONTSIZE = 8
ANNOTATION_FONTSIZE = 9
USE_PERCENTILE_WINDOW = True
PERCENTILE_RANGE = (1, 99)
SAVE_DPI = 200

# Create the output folder if needed
OUTPUT_FIG_DIR.mkdir(parents=True, exist_ok=True)

def list_nifti_files(scan_dir: Path, pattern: str = "*.nii*"):
    """Return all NIfTI files in sorted order."""
    files = sorted(scan_dir.glob(pattern))
    return [fp for fp in files if fp.is_file()]

def strip_nii_suffix(name: str) -> str:
    """Remove .nii or .nii.gz for clean output naming."""
    if name.endswith(".nii.gz"):
        return name[:-7]
    if name.endswith(".nii"):
        return name[:-4]
    return Path(name).stem

def load_3d_volume(path: Path):
    """
    Load a NIfTI file and return a 3D NumPy array plus the nibabel image object.

    If a file is 4D with a singleton last dimension, that singleton is removed.
    If a file is 4D with multiple volumes, the first volume is used for QC export.
    """
    img = nib.load(str(path))
    data = img.get_fdata()

    if data.ndim == 4:
        if data.shape[-1] == 1:
            data = data[..., 0]
        else:
            data = data[..., 0]

    if data.ndim != 3:
        raise ValueError(f"Expected a 3D image after loading, but got shape {data.shape} for {path}")

    return data, img

def get_center_slices(vol: np.ndarray, downsample: int = 1):
    """
    Extract central axial, coronal, and sagittal slices.

    Returns 2D arrays already rotated for more intuitive display.
    """
    x, y, z = vol.shape
    cx, cy, cz = x // 2, y // 2, z // 2

    axial = np.rot90(vol[:, :, cz])[::downsample, ::downsample]
    coronal = np.rot90(vol[:, cy, :])[::downsample, ::downsample]
    sagittal = np.rot90(vol[cx, :, :])[::downsample, ::downsample]

    return axial, coronal, sagittal

def robust_window(slices, use_percentile=True, percentile_range=(1, 99)):
    """
    Compute display intensity limits from the provided slices.
    Using percentiles makes the QC figures more readable across subjects.
    """
    stack = np.concatenate([np.asarray(s).ravel() for s in slices])
    stack = stack[np.isfinite(stack)]

    if stack.size == 0:
        return 0.0, 1.0

    if use_percentile:
        vmin, vmax = np.percentile(stack, percentile_range)
    else:
        vmin, vmax = float(stack.min()), float(stack.max())

    if vmax <= vmin:
        vmax = vmin + 1.0

    return float(vmin), float(vmax)

def prepare_subject_qc(scan_path: Path):
    """
    Load one scan and prepare the three display slices and annotations.
    """
    vol, img = load_3d_volume(scan_path)
    axial, coronal, sagittal = get_center_slices(vol, downsample=DOWNSAMPLE)
    vmin, vmax = robust_window(
        [axial, coronal, sagittal],
        use_percentile=USE_PERCENTILE_WINDOW,
        percentile_range=PERCENTILE_RANGE,
    )

    return {
        "scan_path": scan_path,
        "scan_name": scan_path.name,
        "scan_stem": strip_nii_suffix(scan_path.name),
        "shape": vol.shape,
        "axcodes": nib.aff2axcodes(img.affine),
        "slices": [axial, coronal, sagittal],
        "vmin": vmin,
        "vmax": vmax,
    }

def save_and_display_qc_page(page_subjects, page_index: int, total_pages: int, output_dir: Path):
    """
    Render one contact-sheet figure containing up to SUBJECTS_PER_FIGURE subjects,
    save it as PNG and PDF, and display it directly in the notebook.
    """
    n_rows = len(page_subjects)
    fig_height = max(3.0, ROW_HEIGHT * n_rows)
    fig, axes = plt.subplots(n_rows, 3, figsize=(FIGURE_WIDTH, fig_height))

    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    column_titles = ["Axial (center)", "Coronal (center)", "Sagittal (center)"]

    for row_idx, subject in enumerate(page_subjects):
        for col_idx, panel in enumerate(subject["slices"]):
            ax = axes[row_idx, col_idx]
            ax.imshow(panel, cmap="gray", vmin=subject["vmin"], vmax=subject["vmax"], interpolation="nearest")
            ax.axis("off")

            if row_idx == 0:
                ax.set_title(column_titles[col_idx], fontsize=COLUMN_TITLE_FONTSIZE)

        row_label = (
            f"{subject['scan_name']}
"
            f"shape={subject['shape']} | axcodes={subject['axcodes']}"
        )
        axes[row_idx, 0].set_ylabel(
            row_label,
            rotation=0,
            ha="right",
            va="center",
            fontsize=ROW_LABEL_FONTSIZE,
            labelpad=55,
        )

    fig.suptitle(
        f"Orientation QC contact sheet: page {page_index} / {total_pages}",
        fontsize=TITLE_FONTSIZE,
        y=0.995,
    )

    start_name = page_subjects[0]["scan_stem"]
    end_name = page_subjects[-1]["scan_stem"]
    fig.text(
        0.5,
        0.008,
        f"Subjects on this page: {len(page_subjects)} | First: {start_name} | Last: {end_name}",
        ha="center",
        fontsize=ANNOTATION_FONTSIZE,
    )

    fig.tight_layout(rect=[0.18, 0.03, 1, 0.975])

    png_path = output_dir / f"orientation_qc_page_{page_index:03d}.png"
    pdf_path = output_dir / f"orientation_qc_page_{page_index:03d}.pdf"

    fig.savefig(png_path, dpi=SAVE_DPI, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    return png_path, pdf_path

# =========================
# Run batch export in pages
# =========================
files = list_nifti_files(SCAN_DIR, FILE_PATTERN)
if not files:
    raise FileNotFoundError(f"No NIfTI files found in: {SCAN_DIR}")

print(f"Found {len(files)} scan(s) in: {SCAN_DIR}")
print(f"Saving QC figures to: {OUTPUT_FIG_DIR}")
print(f"Subjects per figure: {SUBJECTS_PER_FIGURE}")

prepared = []
for i, scan_path in enumerate(files, start=1):
    prepared.append(prepare_subject_qc(scan_path))
    print(f"Prepared [{i}/{len(files)}]: {scan_path.name}")

total_pages = math.ceil(len(prepared) / SUBJECTS_PER_FIGURE)

for page_idx in range(total_pages):
    start = page_idx * SUBJECTS_PER_FIGURE
    end = min((page_idx + 1) * SUBJECTS_PER_FIGURE, len(prepared))
    page_subjects = prepared[start:end]

    png_path, pdf_path = save_and_display_qc_page(
        page_subjects=page_subjects,
        page_index=page_idx + 1,
        total_pages=total_pages,
        output_dir=OUTPUT_FIG_DIR,
    )

    print(f"Displayed and saved page {page_idx + 1}/{total_pages}")
    print(f"  PNG: {png_path}")
    print(f"  PDF: {pdf_path}")

print("
Done. Review the exported QC figures and identify which scans need manual reorientation.")



## Step 2. Identify problematic scans and move or copy them to a dedicated folder

After Step 1, review the exported QC figures and decide which scans do **not** align in orientation with the template or the majority of the dataset.

- Create a dedicated folder such as:
  - `/path/to/problematic_scans_for_reorientation`
- **Copy** the problematic scans there first, rather than moving the only original copy.
- Use that dedicated folder as the **input folder** for Step 3.

This keeps the manual reorientation step focused only on the scans that actually need intervention.


## Step 3. Interactive manual reorientation using a template reference

This step opens one scan at a time and shows it **side by side with the template for reference**.

### Interaction keys
- **`a`**: flip LR + AP
- **`d`**: flip LR + SI
- **`s`**: save the corrected scan and continue to the next one
- **`e`**: stop and resume later

### What gets recorded
A CSV file is maintained to track:
- file name
- subject ID
- whether the scan has been reoriented
- whether AP parity was toggled
- whether VD parity was toggled


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Interactive reorientation with template-locked header + side-by-side reference
=============================================================================

Keys
----
- 'a': flip LR+AP (180° about VD axis)
- 'd': flip LR+SI (180° about AP axis)
- 's': save to the output folder using the template affine/header, then continue
- 'e': pause so the session can be resumed later

Behavior
--------
- The template is loaded once and provides the affine/header for all saved outputs.
- Flips are applied to the data array only.
- A CSV file tracks which scans were already processed.
"""

from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib import gridspec
from IPython.display import clear_output

# =========================
# Paths to edit
# =========================
IN_DIR = Path("/path/to/problematic/scans/to/reorient")
OUT_DIR = Path("/path/to/output/reoriented_scans")
CSV_PATH = Path("/path/to/output/scan_reorientation_info.csv")
TEMPLATE_PATH = Path("/path/to/reference/template.nii.gz")

OUT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)

# =========================
# Helper functions
# =========================

def list_nifti_files(in_dir: Path):
    """Return all NIfTI files in sorted order."""
    return sorted([*in_dir.glob("*.nii.gz"), *in_dir.glob("*.nii")])

def strip_nii_suffix(name: str) -> str:
    """Remove .nii or .nii.gz from a filename."""
    if name.endswith(".nii.gz"):
        return name[:-7]
    if name.endswith(".nii"):
        return name[:-4]
    return Path(name).stem

def parse_subject_id(file_name: str) -> str:
    """
    Derive a simple subject ID from the filename.
    Adjust this if your filename convention is different.
    """
    stem = strip_nii_suffix(Path(file_name).name)
    parts = stem.split("_")
    return "_".join(parts[:2]) if len(parts) >= 2 else stem

def ensure_csv(files):
    """
    Create or update the tracking CSV.

    Columns:
    - file name
    - subject ID
    - reoriented
    - AP_opposite
    - VD_opposite
    """
    cols = ["file name", "subject ID", "reoriented", "AP_opposite", "VD_opposite"]

    if CSV_PATH.exists():
        df = pd.read_csv(CSV_PATH)

        for c in cols:
            if c not in df.columns:
                df[c] = "" if c in ["file name", "subject ID", "reoriented"] else 0

        existing = set(df["file name"].astype(str))
        new_rows = [
            {
                "file name": p.name,
                "subject ID": parse_subject_id(p.name),
                "reoriented": "",
                "AP_opposite": 0,
                "VD_opposite": 0,
            }
            for p in files
            if p.name not in existing
        ]

        if new_rows:
            df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

        df = df[df["file name"].isin([p.name for p in files])].reset_index(drop=True)

    else:
        df = pd.DataFrame(
            [
                {
                    "file name": p.name,
                    "subject ID": parse_subject_id(p.name),
                    "reoriented": "",
                    "AP_opposite": 0,
                    "VD_opposite": 0,
                }
                for p in files
            ],
            columns=cols,
        )

    df["reoriented"] = df["reoriented"].astype(str)
    df["AP_opposite"] = df["AP_opposite"].astype(int)
    df["VD_opposite"] = df["VD_opposite"].astype(int)
    df.to_csv(CSV_PATH, index=False)
    return df

def save_csv(df):
    """Write the current tracking table to disk."""
    df.to_csv(CSV_PATH, index=False)

def robust_vmin_vmax(vol):
    """Robust display range for visualization."""
    finite = np.isfinite(vol)
    if not np.any(finite):
        return 0.0, 1.0
    v = vol[finite]
    p1, p99 = np.percentile(v, [1, 99])
    if p99 <= p1:
        return float(v.min()), float(v.max())
    return float(p1), float(p99)

def show_ref_and_scan(ref_vol, vol, title=""):
    """
    Show the template and the current scan side by side.

    Layout:
    - row 1: axial
    - row 2: coronal
    - row 3: sagittal
    """
    assert ref_vol.ndim == 3 and vol.ndim == 3

    rx, ry, rz = ref_vol.shape
    x, y, z = vol.shape
    rcx, rcy, rcz = rx // 2, ry // 2, rz // 2
    cx, cy, cz = x // 2, y // 2, z // 2

    rvmin, rvmax = robust_vmin_vmax(ref_vol)
    vmin, vmax = robust_vmin_vmax(vol)

    plt.figure(figsize=(12, 9))
    gs = gridspec.GridSpec(3, 2, wspace=0.02, hspace=0.04)

    # Axial
    ax = plt.subplot(gs[0, 0])
    ax.imshow(ref_vol[:, :, rcz].T, origin="lower", vmin=rvmin, vmax=rvmax)
    ax.set_title("Template: Axial")
    ax.axis("off")

    ax = plt.subplot(gs[0, 1])
    ax.imshow(vol[:, :, cz].T, origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title("Scan: Axial")
    ax.axis("off")

    # Coronal
    ax = plt.subplot(gs[1, 0])
    ax.imshow(ref_vol[:, rcy, :].T, origin="lower", vmin=rvmin, vmax=rvmax)
    ax.set_title("Template: Coronal")
    ax.axis("off")

    ax = plt.subplot(gs[1, 1])
    ax.imshow(vol[:, cy, :].T, origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title("Scan: Coronal")
    ax.axis("off")

    # Sagittal
    ax = plt.subplot(gs[2, 0])
    ax.imshow(ref_vol[rcx, :, :].T, origin="lower", vmin=rvmin, vmax=rvmax)
    ax.set_title("Template: Sagittal")
    ax.axis("off")

    ax = plt.subplot(gs[2, 1])
    ax.imshow(vol[cx, :, :].T, origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title("Scan: Sagittal")
    ax.axis("off")

    plt.suptitle(title, y=0.98)
    plt.show()

# =========================
# Flip operations
# =========================

def flip_lr_ap(vol):
    """Flip LR (axis 0) and AP (axis 1)."""
    return np.flip(np.flip(vol, axis=0), axis=1)

def flip_lr_si(vol):
    """Flip LR (axis 0) and SI (axis 2)."""
    return np.flip(np.flip(vol, axis=0), axis=2)

# =========================
# I/O helpers
# =========================

def load_scan_array(path: Path):
    """Load one scan as a 3D array and keep the original on-disk dtype."""
    img = nib.load(str(path))
    data = img.get_fdata()

    if data.ndim == 4 and data.shape[-1] == 1:
        data = data[..., 0]

    if data.ndim != 3:
        raise ValueError(f"Only 3D images are supported; got {data.shape} for {path}")

    orig_dtype = img.get_data_dtype()
    return data, orig_dtype

def load_template(path: Path):
    """
    Load the reference template once.

    Returns:
    - template data
    - template affine
    - template header
    - template sform code
    - template qform code
    """
    img = nib.load(str(path))
    data = img.get_fdata()

    if data.ndim == 4 and data.shape[-1] == 1:
        data = data[..., 0]

    if data.ndim != 3:
        raise ValueError(f"Template must be 3D; got {data.shape}")

    affine = img.affine.copy()
    header = img.header.copy()

    try:
        sform_code = int(header["sform_code"])
    except Exception:
        sform_code = 1

    try:
        qform_code = int(header["qform_code"])
    except Exception:
        qform_code = 1

    return data, affine, header, sform_code, qform_code

def save_with_template_header(path_out: Path, vol, tpl_affine, tpl_header, dtype, tpl_scode, tpl_qcode):
    """
    Save the reoriented volume using the template affine/header.

    Notes
    -----
    - The header shape is updated to match the current volume.
    - The sform/qform are set from the template affine.
    """
    hdr = tpl_header.copy()
    hdr.set_data_shape(vol.shape)

    out = nib.Nifti1Image(vol.astype(dtype, copy=False), tpl_affine, header=hdr)
    out.set_sform(tpl_affine, code=int(tpl_scode) if tpl_scode else 1)
    out.set_qform(tpl_affine, code=int(tpl_qcode) if tpl_qcode else 1)
    nib.save(out, str(path_out))

# =========================
# Interactive processing
# =========================

def process_one_file(row, ref_vol, tpl_affine, tpl_header, tpl_scode, tpl_qcode):
    """
    Interactive loop for a single file.

    Returns
    -------
    saved : bool
        True if the user saved and continued.
    ap_parity : int
        Whether the AP-related flip was toggled an odd number of times.
    vd_parity : int
        Whether the VD-related flip was toggled an odd number of times.
    """
    fname = row["file name"]
    in_path = IN_DIR / fname

    if not in_path.exists():
        print(f"[WARN] Missing file: {in_path}")
        return False, row["AP_opposite"], row["VD_opposite"]

    vol, orig_dtype = load_scan_array(in_path)

    ap_parity = 0
    vd_parity = 0

    while True:
        clear_output(wait=True)
        title = f"{fname}  |  a(AP)={ap_parity}  d(VD)={vd_parity}"
        show_ref_and_scan(ref_vol=ref_vol, vol=vol, title=title)

        print("Keys: 'a' (flip LR+AP), 'd' (flip LR+SI), 's' (save & next), 'e' (pause), 'help'")
        cmd = input("Enter key: ").strip().lower()

        if cmd == "a":
            vol = flip_lr_ap(vol)
            ap_parity ^= 1
            continue

        elif cmd == "d":
            vol = flip_lr_si(vol)
            vd_parity ^= 1
            continue

        elif cmd == "s":
            out_path = OUT_DIR / fname
            save_with_template_header(
                out_path, vol, tpl_affine, tpl_header, orig_dtype, tpl_scode, tpl_qcode
            )
            print(f"[OK] Saved with template header/affine: {out_path}")
            return True, ap_parity, vd_parity

        elif cmd == "e":
            print("[STOP] Pausing on your request.")
            return False, ap_parity, vd_parity

        elif cmd in ("help", "?"):
            print("a  -> flip LR and AP")
            print("d  -> flip LR and SI")
            print("s  -> save using the template header/affine, then continue")
            print("e  -> stop now and resume later")
            _ = input("Press Enter to continue...")
            continue

        else:
            print(f"[INFO] Unrecognized key: {cmd}")
            _ = input("Press Enter to continue...")
            continue

def main():
    """Main driver for the interactive manual reorientation workflow."""
    if not TEMPLATE_PATH.exists():
        raise FileNotFoundError(f"Template not found: {TEMPLATE_PATH}")

    ref_vol, tpl_affine, tpl_header, tpl_scode, tpl_qcode = load_template(TEMPLATE_PATH)

    files = list_nifti_files(IN_DIR)
    if not files:
        print(f"No NIfTI files found in {IN_DIR}")
        return

    df = ensure_csv(files)

    pending_mask = df["reoriented"].str.lower().ne("y")
    if not pending_mask.any():
        print("All rows are already marked as reoriented. Nothing to do.")
        return

    for idx in df.index[pending_mask]:
        row = df.loc[idx].copy()
        print(f"\n=== Processing {row['file name']} (subject: {row['subject ID']}) ===")

        saved, ap_parity, vd_parity = process_one_file(
            row, ref_vol, tpl_affine, tpl_header, tpl_scode, tpl_qcode
        )

        if not saved:
            save_csv(df)
            print("\nProgress saved. Rerun this cell later to resume.")
            return

        df.at[idx, "reoriented"] = "y"
        df.at[idx, "AP_opposite"] = int(ap_parity)
        df.at[idx, "VD_opposite"] = int(vd_parity)
        save_csv(df)

    print("\nAll pending scans processed. CSV updated and outputs written.")

# Run
if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print("[ERROR]", e)
        raise
